In [0]:
import pandas as pd

In [0]:
# Cell 1
%pip install -U mlflow
dbutils.library.restartPython()

# Cell 2 (after restart)
import mlflow
mlflow.openai.autolog()  # auto-logs all your LLM calls — free tracing!

from openai import OpenAI

# This works inside any Databricks notebook
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get() + "/serving-endpoints"
)

# Quick test to confirm it works
print("Client ready!")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Client ready!


In [0]:
# Cell 3 - Define the schema
from pydantic import BaseModel, Field
from typing import Optional, List

class FacilityRecord(BaseModel):
    has_icu: bool = False
    has_ot: bool = False
    has_nicu: bool = False
    has_dialysis: bool = False
    has_oncology: bool = False
    has_blood_bank: bool = False
    has_oxygen_supply: bool = False
    is_24_7: bool = False
    has_anesthesiologist: bool = False
    has_surgeon: bool = False
    doctor_count: Optional[int] = None
    uses_parttime_doctors: bool = False
    equipment_mentioned: List[str] = Field(default_factory=list)
    specialties_mentioned: List[str] = Field(default_factory=list)

print("Schema ready!")

Schema ready!


In [0]:
# Cell 1 - Always run this first
%pip install -U mlflow openai pydantic pandas openpyxl


# Cell 2 - Restart happens after pip install, so redefine everything here
import mlflow
from openai import OpenAI

mlflow.openai.autolog()

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get() + "/serving-endpoints"
)

print("Client ready!")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Client ready!


In [0]:
# Run this to see all available models in your workspace
import requests

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

response = requests.get(
    f"{host}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {token}"}
)

endpoints = response.json().get("endpoints", [])
for e in endpoints:
    print(e["name"])

databricks-gpt-5-4
databricks-gpt-5-5
databricks-gpt-5-5-pro
databricks-gpt-5-4-mini
databricks-gpt-5-4-nano
databricks-gpt-5-2
databricks-gpt-oss-120b
databricks-gpt-5-3-codex
databricks-gpt-5-2-codex
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-gpt-5-1
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-meta-llama-3.1-405b-instruct


In [0]:
# Cell 4 - Test with one fake note
test_note = """
District hospital Patna. Has functional ICU with 6 beds and oxygen supply.
Operation theatre available. Anesthesiologist on staff. Open 24/7.
Blood bank on premises. 4 full-time doctors.
"""

response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",  # updated
    messages=[{
        "role": "user",
        "content": f"""Extract medical facility info as JSON only, no explanation:
        
{test_note}

Return only this JSON structure:
{{
  "has_icu": true/false,
  "has_ot": true/false,
  "has_blood_bank": true/false,
  "has_oxygen_supply": true/false,
  "is_24_7": true/false,
  "has_anesthesiologist": true/false,
  "doctor_count": number or null,
  "equipment_mentioned": [],
  "specialties_mentioned": []
}}"""
    }],
    max_tokens=300,
    temperature=0.0
)

print(response.choices[0].message.content)

```
{
  "has_icu": true,
  "has_ot": true,
  "has_blood_bank": true,
  "has_oxygen_supply": true,
  "is_24_7": true,
  "has_anesthesiologist": true,
  "doctor_count": 4,
  "equipment_mentioned": [],
  "specialties_mentioned": []
}
```


Trace(trace_id=tr-07e9afdfc8240032a0d5da10d5f0a011)

In [0]:
MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-bge-large-en"  # for vector search later

In [0]:
import pandas as pd

df = pd.read_csv("/Workspace/Users/eimansaeed1707@gmail.com/serving_nation/HACKATHON_DATA.csv")
print(df.shape)

(10000, 41)


In [0]:
print(df.columns.tolist())


['name', 'phone_numbers', 'officialPhone', 'email', 'websites', 'officialWebsite', 'yearEstablished', 'facebookLink', 'twitterLink', 'linkedinLink', 'instagramLink', 'address_line1', 'address_line2', 'address_line3', 'address_city', 'address_stateOrRegion', 'address_zipOrPostcode', 'address_country', 'address_countryCode', 'facilityTypeId', 'operatorTypeId', 'affiliationTypeIds', 'description', 'numberDoctors', 'capacity', 'specialties', 'procedure', 'equipment', 'capability', 'recency_of_page_update', 'distinct_social_media_presence_count', 'affiliated_staff_presence', 'custom_logo_presence', 'number_of_facts_about_the_organization', 'post_metrics_most_recent_social_media_post_date', 'post_metrics_post_count', 'engagement_metrics_n_followers', 'engagement_metrics_n_likes', 'engagement_metrics_n_engagements', 'latitude', 'longitude']


In [0]:
# Cell 6 - Check what the text columns actually look like
sample = df.iloc[0]
print("NAME:", sample['name'])
print("\nDESCRIPTION:", sample['description'])
print("\nSPECIALTIES:", sample['specialties'])
print("\nEQUIPMENT:", sample['equipment'])
print("\nCAPABILITY:", sample['capability'])
print("\nPROCEDURE:", sample['procedure'])
print("\nDOCTORS:", sample['numberDoctors'])
print("\nLAT/LNG:", sample['latitude'], sample['longitude'])

NAME: 1000 Smiles Dental Clinic

DESCRIPTION: Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.

SPECIALTIES: ["familyMedicine","periodontics","endodontics","dentistry","aestheticDentistry"]

EQUIPMENT: []

CAPABILITY: ["Has been in operation for 10 years","Dental clinic"]

PROCEDURE: ["Performs root canal therapy (RCT)","Provides laser dentistry services"]

DOCTORS: nan

LAT/LNG: 17.39773941 78.48268127


In [0]:
import json
import pandas as pd

def parse_list(val):
    if val is None:
        return []
    if isinstance(val, bool):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, float):
        return []
    text = str(val).strip()
    if text in ['', '[]', 'nan']:
        return []
    try:
        result = json.loads(text.replace("'", '"'))
        if isinstance(result, list):
            return result
        return [str(result)]
    except:
        return [text]

def safe_check(term, text):
    try:
        return term.lower() in str(text).lower()
    except:
        return False

def extract_facility(row):
    specialties = parse_list(row['specialties'])
    equipment   = parse_list(row['equipment'])
    capability  = parse_list(row['capability'])
    procedure   = parse_list(row['procedure'])

    # Combine all text into one searchable string
    notes = " ".join([
        str(row.get('description', '')),
        " ".join(specialties),
        " ".join(equipment),
        " ".join(capability),
        " ".join(procedure)
    ]).lower()

    # Doctor count
    try:
        doc_count = int(row['numberDoctors']) if not pd.isna(row['numberDoctors']) else None
    except:
        doc_count = None

    return {
        'facility_id':            str(row.name),
        'name':                   str(row.get('name', '')),
        'district':               str(row.get('address_city', '')),
        'state':                  str(row.get('address_stateOrRegion', '')),
        'pin_code':               str(row.get('address_zipOrPostcode', '')),
        'latitude':               row.get('latitude', None),
        'longitude':              row.get('longitude', None),
        'raw_notes':              notes,

        'has_icu':                safe_check('icu', notes) or safe_check('intensive care', notes),
        'has_ot':                 safe_check('operation theatre', notes) or safe_check('surgical suite', notes),
        'has_nicu':               safe_check('nicu', notes) or safe_check('neonatal', notes),
        'has_dialysis':           safe_check('dialysis', notes),
        'has_oncology':           safe_check('oncol', notes) or safe_check('cancer', notes) or safe_check('chemotherapy', notes),
        'has_blood_bank':         safe_check('blood bank', notes),
        'has_oxygen_supply':      safe_check('oxygen', notes),
        'is_24_7':                safe_check('24/7', notes) or safe_check('24 hours', notes) or safe_check('round the clock', notes),
        'has_anesthesiologist':   safe_check('anesthes', notes) or safe_check('anaesthes', notes),
        'has_surgeon':            safe_check('surgeon', notes),
        'doctor_count':           doc_count,
        'uses_parttime_doctors':  safe_check('part-time', notes) or safe_check('part time', notes) or safe_check('visiting doctor', notes),
        'equipment_mentioned':    equipment,
        'specialties_mentioned':  specialties,
        'data_staleness_years':   row.get('recency_of_page_update', None)
    }

# Test on first 3 rows
for i in range(3):
    result = extract_facility(df.iloc[i])
    print(f"\n--- Row {i}: {result['name']} ---")
    for k, v in result.items():
        if k not in ['raw_notes']:  # skip raw notes to keep output clean
            print(f"  {k}: {v}")


--- Row 0: 1000 Smiles Dental Clinic ---
  facility_id: 0
  name: 1000 Smiles Dental Clinic
  district: Hyderabad
  state: Telangana
  pin_code: 500013
  latitude: 17.39773941
  longitude: 78.48268127
  has_icu: False
  has_ot: False
  has_nicu: False
  has_dialysis: False
  has_oncology: False
  has_blood_bank: False
  has_oxygen_supply: False
  is_24_7: False
  has_anesthesiologist: False
  has_surgeon: False
  doctor_count: None
  uses_parttime_doctors: False
  equipment_mentioned: []
  specialties_mentioned: ['familyMedicine', 'periodontics', 'endodontics', 'dentistry', 'aestheticDentistry']
  data_staleness_years: nan

--- Row 1: 108 Eye And Heath Centre ---
  facility_id: 1
  name: 108 Eye And Heath Centre
  district: Noida
  state: Uttar Pradesh
  pin_code: 201307
  latitude: 28.57222366
  longitude: 77.36903381
  has_icu: False
  has_ot: False
  has_nicu: False
  has_dialysis: False
  has_oncology: False
  has_blood_bank: False
  has_oxygen_supply: False
  is_24_7: False
  has

In [0]:
# Process all 10,000 rows
results = []
errors = []

for i, row in df.iterrows():
    try:
        record = extract_facility(row)
        results.append(record)
    except Exception as e:
        errors.append({'row': i, 'error': str(e)})
    
    if i % 500 == 0:
        print(f"Progress: {i}/10000...")

extracted_df = pd.DataFrame(results)
print(f"\nDone!")
print(f"Extracted: {len(results)}")
print(f"Errors: {len(errors)}")
print(f"\nSample trust-relevant counts:")
print(f"  Has ICU: {extracted_df['has_icu'].sum()}")
print(f"  Has OT: {extracted_df['has_ot'].sum()}")
print(f"  Has Dialysis: {extracted_df['has_dialysis'].sum()}")
print(f"  Is 24/7: {extracted_df['is_24_7'].sum()}")
print(f"  Has Blood Bank: {extracted_df['has_blood_bank'].sum()}")

Progress: 0/10000...
Progress: 500/10000...
Progress: 1000/10000...
Progress: 1500/10000...
Progress: 2000/10000...
Progress: 2500/10000...
Progress: 3000/10000...
Progress: 3500/10000...
Progress: 4000/10000...
Progress: 4500/10000...
Progress: 5000/10000...
Progress: 5500/10000...
Progress: 6000/10000...
Progress: 6500/10000...
Progress: 7000/10000...
Progress: 7500/10000...
Progress: 8000/10000...
Progress: 8500/10000...
Progress: 9000/10000...
Progress: 9500/10000...

Done!
Extracted: 10000
Errors: 0

Sample trust-relevant counts:
  Has ICU: 299
  Has OT: 56
  Has Dialysis: 45
  Is 24/7: 776
  Has Blood Bank: 23


In [0]:
# Validator Agent - adds Trust Score to every record

POSITIVE_SIGNALS = {
    'has_icu': 2,
    'has_ot': 1.5,
    'has_blood_bank': 1,
    'has_oxygen_supply': 1,
    'has_anesthesiologist': 1,
    'has_surgeon': 1,
    'is_24_7': 1
}

MEDICAL_RULES = {
    'ot_without_anesthesiologist': {
        'condition': lambda r: r['has_ot'] and not r['has_anesthesiologist'],
        'penalty': 3,
        'message': 'Claims OT but no anesthesiologist listed'
    },
    'icu_without_oxygen': {
        'condition': lambda r: r['has_icu'] and not r['has_oxygen_supply'],
        'penalty': 2,
        'message': 'Claims ICU but no oxygen supply mentioned'
    },
    '24_7_without_doctors': {
        'condition': lambda r: r['is_24_7'] and (r['doctor_count'] or 0) < 2,
        'penalty': 2,
        'message': 'Claims 24/7 but fewer than 2 doctors listed'
    },
    'nicu_without_neonatologist': {
        'condition': lambda r: r['has_nicu'] and 'neonatologist' not in [s.lower() for s in r['specialties_mentioned']],
        'penalty': 1.5,
        'message': 'Claims NICU but no neonatologist found'
    },
    'dialysis_without_nephrologist': {
        'condition': lambda r: r['has_dialysis'] and 'nephrology' not in [s.lower() for s in r['specialties_mentioned']],
        'penalty': 1,
        'message': 'Claims dialysis but no nephrologist found'
    }
}

def calculate_trust_score(record):
    score = 5.0
    contradictions = []

    for field, bonus in POSITIVE_SIGNALS.items():
        if record.get(field):
            score += bonus

    for rule_name, rule in MEDICAL_RULES.items():
        try:
            if rule['condition'](record):
                score -= rule['penalty']
                contradictions.append(rule['message'])
        except:
            pass

    score = max(0, min(10, score))

    if score >= 7:   level = 'HIGH'
    elif score >= 5: level = 'MEDIUM'
    elif score >= 3: level = 'LOW'
    else:            level = 'SUSPICIOUS'

    return {
        'trust_score': round(score, 1),
        'trust_level': level,
        'contradictions': contradictions,
        'trust_reasoning': f"Score {score:.1f}/10. Issues: {'; '.join(contradictions) if contradictions else 'None'}"
    }

# Run on all 10,000 records
for record in results:
    trust = calculate_trust_score(record)
    record.update(trust)

validated_df = pd.DataFrame(results)

print("Trust Score Distribution:")
print(validated_df['trust_level'].value_counts())
print(f"\nAverage trust score: {validated_df['trust_score'].mean():.2f}")
print(f"\nMost suspicious facilities:")
print(validated_df[validated_df['trust_level'] == 'SUSPICIOUS'][['name', 'trust_score', 'trust_reasoning']].head(5))

Trust Score Distribution:
trust_level
MEDIUM        9181
LOW            751
SUSPICIOUS      35
HIGH            33
Name: count, dtype: int64

Average trust score: 4.97

Most suspicious facilities:
                                             name  ...                                    trust_reasoning
174                      Aastha Children Hospital  ...  Score 2.5/10. Issues: Claims ICU but no oxygen...
264                               Acadis Hospital  ...  Score 2.5/10. Issues: Claims ICU but no oxygen...
277   Ace Nx Hospital and Research Center, Kalyan  ...  Score 2.5/10. Issues: Claims ICU but no oxygen...
326          Advance New Born & Child Care Centre  ...  Score 2.5/10. Issues: Claims ICU but no oxygen...
1154    Ashirwad Urology and Laparoscopy Hospital  ...  Score 2.5/10. Issues: Claims OT but no anesthe...

[5 rows x 3 columns]


In [0]:
# Save validated data
validated_df.to_csv("/Workspace/Users/eimansaeed1707@gmail.com/serving_nation/validated.csv", index=False)
print("Saved!")

# Medical Desert Score by city
desert_stats = validated_df.groupby('district').agg(
    facility_count=('facility_id', 'count'),
    avg_trust_score=('trust_score', 'mean'),
    has_any_icu=('has_icu', 'max'),
    has_any_ot=('has_ot', 'max'),
    has_any_dialysis=('has_dialysis', 'max'),
    has_any_blood_bank=('has_blood_bank', 'max'),
    avg_lat=('latitude', 'mean'),
    avg_lng=('longitude', 'mean')
).reset_index()

def desert_score(row):
    score = 0
    if row['facility_count'] < 3:  score += 30
    elif row['facility_count'] < 5: score += 15
    if row['avg_trust_score'] < 5:  score += 20
    if not row['has_any_icu']:      score += 20
    if not row['has_any_ot']:       score += 15
    if not row['has_any_blood_bank']: score += 10
    if not row['has_any_dialysis']: score += 5
    return min(score, 100)

desert_stats['desert_score'] = desert_stats.apply(desert_score, axis=1)

print("\nTop 10 Medical Deserts:")
print(desert_stats.nlargest(10, 'desert_score')[['district', 'facility_count', 'avg_trust_score', 'desert_score']])

Saved!

Top 10 Medical Deserts:
          district  facility_count  avg_trust_score  desert_score
19    Ahilya Nagar               1             4.00           100
49      Amalapuram               2             4.25           100
65       Amleshwar               1             4.00           100
71       Anaimalai               1             4.00           100
78          Anchal               1             4.00           100
100         Araria               1             4.00           100
105         Armoor               2             4.50           100
108  Aruppukkottai               2             4.50           100
115        Assandh               1             4.00           100
123        Attapur               1             4.00           100


In [0]:
%pip install folium
import folium
from folium.plugins import HeatMap

# Base map of India
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="CartoDB positron")

# Heatmap layer - medical deserts
heat_data = desert_stats[['avg_lat', 'avg_lng', 'desert_score']].dropna().values.tolist()
HeatMap(heat_data, name="Medical Deserts", min_opacity=0.4, radius=20).add_to(m)

# Add markers for HIGH trust facilities
high_trust = validated_df[validated_df['trust_level'] == 'HIGH'].dropna(subset=['latitude', 'longitude'])
for _, row in high_trust.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='#1a9e6a',
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(f"""
            <b>{row['name']}</b><br>
            {row['district']}, {row['state']}<br>
            Trust: {row['trust_score']}/10 ({row['trust_level']})<br>
            ICU: {'Yes' if row['has_icu'] else 'No'} |
            OT: {'Yes' if row['has_ot'] else 'No'} |
            24/7: {'Yes' if row['is_24_7'] else 'No'}<br>
            <small>{row['trust_reasoning']}</small>
        """, max_width=300)
    ).add_to(m)

# Add markers for SUSPICIOUS facilities
suspicious = validated_df[validated_df['trust_level'] == 'SUSPICIOUS'].dropna(subset=['latitude', 'longitude'])
for _, row in suspicious.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='#e63946',
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(f"""
            <b>{row['name']}</b><br>
            {row['district']}, {row['state']}<br>
            Trust: {row['trust_score']}/10 (SUSPICIOUS)<br>
            ⚠️ {row['trust_reasoning']}
        """, max_width=300)
    ).add_to(m)

folium.LayerControl().add_to(m)

# Save map
map_path = "/Workspace/Users/eimansaeed1707@gmail.com/serving_nation/india_healthcare_map.html"
m.save(map_path)
print(f"Map saved!")
print(f"Green markers: {len(high_trust)} high-trust facilities")
print(f"Red markers: {len(suspicious)} suspicious facilities")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Map saved!
Green markers: 33 high-trust facilities
Red markers: 35 suspicious facilities


In [0]:
# Query Agent - natural language search over your 10,000 facilities

def query_agent(user_query):
    # Step 1: Filter validated_df based on query keywords
    query_lower = user_query.lower()
    candidates = validated_df.copy()
    
    # Apply relevant filters based on what query mentions
    if any(w in query_lower for w in ['icu', 'intensive care']):
        candidates = candidates[candidates['has_icu'] == True]
    if any(w in query_lower for w in ['surgery', 'operation', 'appendectomy', 'surgical']):
        candidates = candidates[candidates['has_ot'] == True]
    if 'dialysis' in query_lower:
        candidates = candidates[candidates['has_dialysis'] == True]
    if 'nicu' in query_lower or 'neonatal' in query_lower or 'newborn' in query_lower:
        candidates = candidates[candidates['has_nicu'] == True]
    if '24' in query_lower or 'emergency' in query_lower:
        candidates = candidates[candidates['is_24_7'] == True]
    if 'blood' in query_lower:
        candidates = candidates[candidates['has_blood_bank'] == True]
    if 'cancer' in query_lower or 'oncology' in query_lower:
        candidates = candidates[candidates['has_oncology'] == True]
    if 'part time' in query_lower or 'parttime' in query_lower or 'part-time' in query_lower:
        candidates = candidates[candidates['uses_parttime_doctors'] == True]

    # Filter by state/city if mentioned
    for state in validated_df['state'].dropna().unique():
        if state.lower() in query_lower:
            candidates = candidates[candidates['state'] == state]
            break
    for city in validated_df['district'].dropna().unique():
        if city.lower() in query_lower:
            candidates = candidates[candidates['district'] == city]
            break

    # Sort by trust score, take top 20 for LLM
    candidates = candidates.sort_values('trust_score', ascending=False).head(20)

    if len(candidates) == 0:
        return "No facilities found matching your query. Try broader search terms."

    # Step 2: LLM reasons over the candidates
    candidates_text = candidates[[
        'name', 'district', 'state', 'trust_score', 'trust_level',
        'trust_reasoning', 'has_icu', 'has_ot', 'is_24_7',
        'has_blood_bank', 'specialties_mentioned', 'raw_notes'
    ]].to_dict('records')

    # Truncate raw_notes for each candidate
    for c in candidates_text:
        c['raw_notes'] = str(c['raw_notes'])[:300]

    response = client.chat.completions.create(
        model=MODEL,
        # Replace the messages section in query_agent with this

messages=[
    {
        "role": "system",
        "content": """You are a healthcare intelligence agent for India.
You will be given REAL facility records. Your job is to pick the top 3.

STRICT RULES - violations will be penalised:
- Trust score MUST be copied exactly from the 'trust_score' field. Never calculate your own.
- Raw notes quote MUST be copied exactly from the 'raw_notes' field. Never invent quotes.
- If part-time doctors are not confirmed in the data, say "not confirmed in data"
- Do not add any information that is not in the candidate records provided

Format each result exactly like this:
**1. [name]**
- Location: [district], [state]
- Trust Score: [exact trust_score from data]/10 ([trust_level])
- Capabilities matched: [list only what is true in the data]
- Warnings: [trust_reasoning from data]
- Evidence: [copy one sentence directly from raw_notes]"""
    },
    {
        "role": "user",
        "content": f"""Query: {user_query}

Here are the REAL candidate records from our database. Use ONLY this data:

{json.dumps(candidates_text, indent=2)}

Pick top 3. Copy trust scores and quotes exactly. Do not invent anything."""
    } ]


    return response.choices[0].message.content

# Test with the challenge brief example query
result = query_agent("Find the nearest facility in rural Bihar that can perform an emergency appendectomy and typically leverages parttime doctors")
print(result)

In [0]:
# Check what states exist in the data
print("States in dataset:")
print(validated_df['state'].value_counts().head(20))

print("\nFacilities with OT:")
print(validated_df[validated_df['has_ot'] == True]['state'].value_counts().head(10))

print("\nFacilities with part-time doctors:")
print(validated_df[validated_df['uses_parttime_doctors'] == True]['state'].value_counts().head(10))

States in dataset:
state
Maharashtra       1506
Uttar Pradesh     1058
Gujarat            838
Tamil Nadu         630
Kerala             597
Rajasthan          495
West Bengal        483
Karnataka          455
Delhi              447
Bihar              429
Telangana          429
Haryana            385
Punjab             372
Madhya Pradesh     371
Andhra Pradesh     276
Jharkhand          140
Uttarakhand        136
Assam              126
Chhattisgarh       115
Odisha             109
Name: count, dtype: int64

Facilities with OT:
state
Gujarat           12
Maharashtra        9
Rajasthan          5
Uttar Pradesh      4
Andhra Pradesh     3
Haryana            3
Punjab             3
Kerala             3
Bihar              2
Odisha             2
Name: count, dtype: int64

Facilities with part-time doctors:
state
Gujarat             2
Kerala              1
Himachal Pradesh    1
Name: count, dtype: int64


In [0]:
def query_agent(user_query):
    query_lower = user_query.lower()
    
    # Start with all facilities
    candidates = df.copy()
    candidates['match_score'] = 0.0

    # Hard filter by state or city if mentioned
    for state in validated_df['state'].dropna().unique():
        if state.lower() in query_lower:
            candidates = candidates[candidates['state'] == state]
            break
    for city in validated_df['district'].dropna().unique():
        if city.lower() in query_lower:
            candidates = candidates[candidates['district'] == city]
            break

    # Reward matching capabilities
    if any(w in query_lower for w in ['icu', 'intensive care']):
        candidates.loc[candidates['has_icu'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['surgery', 'operation', 'appendectomy', 'surgical']):
        candidates.loc[candidates['has_ot'] == True, 'match_score'] += 3
    if 'dialysis' in query_lower:
        candidates.loc[candidates['has_dialysis'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['nicu', 'neonatal', 'newborn', 'baby']):
        candidates.loc[candidates['has_nicu'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['24', 'emergency', 'urgent']):
        candidates.loc[candidates['is_24_7'] == True, 'match_score'] += 2
    if 'blood' in query_lower:
        candidates.loc[candidates['has_blood_bank'] == True, 'match_score'] += 2
    if any(w in query_lower for w in ['cancer', 'oncology']):
        candidates.loc[candidates['has_oncology'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['part time', 'parttime', 'part-time', 'visiting']):
        candidates.loc[candidates['uses_parttime_doctors'] == True, 'match_score'] += 2

    # Rank by match + trust score
    candidates['final_score'] = candidates['match_score'] + candidates['trust_score']
    candidates = candidates.sort_values('final_score', ascending=False).head(20)

    print(f"Sending {len(candidates)} candidates to LLM...")

    # Prepare candidates for LLM
    candidates_text = []
    for _, row in candidates.iterrows():
        candidates_text.append({
            'name': str(row['name']),
            'district': str(row['district']),
            'state': str(row['state']),
            'trust_score': float(row['trust_score']),
            'trust_level': str(row['trust_level']),
            'trust_reasoning': str(row['trust_reasoning']),
            'has_icu': bool(row['has_icu']),
            'has_ot': bool(row['has_ot']),
            'is_24_7': bool(row['is_24_7']),
            'has_blood_bank': bool(row['has_blood_bank']),
            'specialties_mentioned': str(row['specialties_mentioned'])[:200],
            'raw_notes': str(row['raw_notes'])[:300],
            'match_score': float(row['match_score']),
            'final_score': float(row['final_score'])
        })

    response = client.chat.completions.create(
        model=MODEL,
# Replace the messages section in query_agent with this

messages=[
    {
        "role": "system",
        "content": """You are a healthcare intelligence agent for India.
You will be given REAL facility records. Your job is to pick the top 3.

STRICT RULES - violations will be penalised:
- Trust score MUST be copied exactly from the 'trust_score' field. Never calculate your own.
- Raw notes quote MUST be copied exactly from the 'raw_notes' field. Never invent quotes.
- If part-time doctors are not confirmed in the data, say "not confirmed in data"
- Do not add any information that is not in the candidate records provided

Format each result exactly like this:
**1. [name]**
- Location: [district], [state]
- Trust Score: [exact trust_score from data]/10 ([trust_level])
- Capabilities matched: [list only what is true in the data]
- Warnings: [trust_reasoning from data]
- Evidence: [copy one sentence directly from raw_notes]"""
    },
    {
        "role": "user",
        "content": f"""Query: {user_query}

Here are the REAL candidate records from our database. Use ONLY this data:

{json.dumps(candidates_text, indent=2)}

Pick top 3. Copy trust scores and quotes exactly. Do not invent anything."""
    }
],
        max_tokens=1000
    )

    return response.choices[0].message.content if hasattr(response.choices[0], 'message') else str(response.choices[0])

# Test
result = query_agent("Find a facility in Bihar for emergency surgery")
print(result)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8040752017474180>, line 103
    100     return response.choices[0].message.content if hasattr(response.choices[0], 'message') else str(response.choices[0])
    102 # Test
--> 103 result = query_agent("Find a facility in Bihar for emergency surgery")
    104 print(result)

File <command-8040752017474180>, line 5, in query_agent(user_query)
      2 query_lower = user_query.lower()
      4 # Start with all facilities
----> 5 candidates = df.copy()
      6 candidates['match_score'] = 0.0
      8 # Hard filter by state or city if mentioned

NameError: name 'df' is not defined

In [0]:
%pip install --upgrade gradio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import json
import gradio as gr
from openai import OpenAI

# Client setup
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get() + "/serving-endpoints"
)
MODEL = "databricks-meta-llama-3-3-70b-instruct"

# Reload data
validated_df = pd.read_csv("/Workspace/Users/eimansaeed1707@gmail.com/serving_nation/validated.csv")
print(f"Loaded {len(validated_df)} rows")

# Query agent
def query_agent(user_query):
    query_lower = user_query.lower()
    candidates = validated_df.copy()
    candidates['match_score'] = 0.0

    for state in validated_df['state'].dropna().unique():
        if state.lower() in query_lower:
            candidates = candidates[candidates['state'] == state]
            break
    for city in validated_df['district'].dropna().unique():
        if city.lower() in query_lower:
            candidates = candidates[candidates['district'] == city]
            break

    if any(w in query_lower for w in ['icu', 'intensive care']):
        candidates.loc[candidates['has_icu'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['surgery', 'operation', 'appendectomy', 'surgical']):
        candidates.loc[candidates['has_ot'] == True, 'match_score'] += 3
    if 'dialysis' in query_lower:
        candidates.loc[candidates['has_dialysis'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['nicu', 'neonatal', 'newborn', 'baby']):
        candidates.loc[candidates['has_nicu'] == True, 'match_score'] += 3
    if any(w in query_lower for w in ['24', 'emergency', 'urgent']):
        candidates.loc[candidates['is_24_7'] == True, 'match_score'] += 2
    if 'blood' in query_lower:
        candidates.loc[candidates['has_blood_bank'] == True, 'match_score'] += 2
    if any(w in query_lower for w in ['cancer', 'oncology']):
        candidates.loc[candidates['has_oncology'] == True, 'match_score'] += 3

    candidates['final_score'] = candidates['match_score'] + candidates['trust_score']
    candidates = candidates.sort_values('final_score', ascending=False).head(20)

    candidates_text = []
    for _, row in candidates.iterrows():
        candidates_text.append({
            'name': str(row['name']),
            'district': str(row['district']),
            'state': str(row['state']),
            'trust_score': float(row['trust_score']),
            'trust_level': str(row['trust_level']),
            'trust_reasoning': str(row['trust_reasoning']),
            'has_icu': bool(row['has_icu']),
            'has_ot': bool(row['has_ot']),
            'is_24_7': bool(row['is_24_7']),
            'specialties_mentioned': str(row['specialties_mentioned'])[:200],
            'raw_notes': str(row['raw_notes'])[:300]
        })

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """You are a healthcare intelligence agent for India.
Return TOP 3 facility matches for the query.
RULES: Only use trust scores from the data. Never invent quotes or scores.
For each facility show: name, location, trust score, matching capabilities, warnings, one line from raw_notes."""
            },
            {
                "role": "user",
                "content": f"Query: {user_query}\n\nCandidates:\n{json.dumps(candidates_text, indent=2)}\n\nReturn top 3 with reasoning."
            }
        ],
        max_tokens=1000
    )

    # Safe response extraction
    try:
        return response.choices[0].message.content
    except:
        try:
            return response['choices'][0]['message']['content']
        except:
            return str(response)

# Gradio interface
def chat(query):
    if not query.strip():
        return "Please enter a query."
    try:
        return query_agent(query)
    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        lines=3,
        placeholder="e.g. Find a 24/7 hospital with ICU in Maharashtra...",
        label="Ask about healthcare facilities in India"
    ),
    outputs=gr.Markdown(label="Results"),
    title="🏥 Serving A Nation — Healthcare Intelligence",
    description="Agentic search across 10,000 Indian medical facilities with Trust Scores",
    examples=[
        ["Find a facility in Bihar for emergency surgery"],
        ["Which hospitals in Kerala have dialysis?"],
        ["Show me high trust ICU facilities in Maharashtra"],
        ["Find a 24/7 hospital with blood bank in Delhi"]
    ]
)

demo.launch(share=True)

Loaded 10000 rows
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://33f92599dbfde14162.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [0]:

# Run this debug cell first
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say the word hello only"}],
    max_tokens=10
)

print(type(test_response))
print(type(test_response.choices))
print(type(test_response.choices[0]))
print(dir(test_response.choices[0]))
print("---")
print(test_response.choices[0])

<class 'openai.types.chat.chat_completion.ChatCompletion'>
<class 'list'>
<class 'openai.types.chat.chat_completion.Choice'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__

Trace(trace_id=tr-204af66289929b74986b8ce31559ba60)

In [0]:
demo.launch(
    share=False,
    server_name="0.0.0.0",
    server_port=7860
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.
